# A small search, then a comparison

**You are here:** hyperparameter search, then model selection on the validation rows. Do not score the test rows.

`tune_forest` tries 8 forests on the train rows, using 3-fold ROC-AUC. The grid is short on purpose: 40, 80, or 120 trees, depth 3, 5, or 8, and at least 5, 10, or 20 rows in a leaf.

If this cell is still running at 14:20, stop it and call `tune_forest` with `n_iter=4`.

Majority ROC-AUC is 0.5 because a constant score does not rank a churner above a stayer. Accuracy can look fine while the ranking is useless. Compare AUC.


In [ ]:
import sys
from pathlib import Path

here = Path.cwd().resolve()
root = None
for candidate in [here, *here.parents]:
    if (candidate / "app.py").is_file() and (candidate / "lib" / "schema.py").is_file():
        root = candidate
        break
    nested = candidate / "01-supervised-learning"
    if (nested / "app.py").is_file() and (nested / "lib" / "schema.py").is_file():
        root = nested
        break
if root is None:
    raise FileNotFoundError(
        "Start Jupyter from the repo folder or from 01-supervised-learning."
    )
lib = root / "lib"
if str(lib) not in sys.path:
    sys.path.insert(0, str(lib))
ROOT = root
print("cwd", here)
print("module", ROOT)


In [ ]:
import pandas as pd

customers = pd.read_csv(ROOT / "data" / "fresh_and_good_customers.csv")
print(customers.shape)
customers.head()


In [ ]:
from frame import features_and_target
from split import split_customers

train, validation, test = split_customers(customers)
print("train", len(train), "validation", len(validation), "test", len(test))
X_train, y_train = features_and_target(train)
X_val, y_val = features_and_target(validation)
X_test, y_test = features_and_target(test)
print("churned is a label, not a feature:", "churned" not in X_train.columns)


In [ ]:
from sklearn.metrics import roc_auc_score

from pipelines import make_forest_pipeline, make_logistic_pipeline, tune_forest

logistic = make_logistic_pipeline()
logistic.fit(X_train, y_train)
forest = make_forest_pipeline()
forest.fit(X_train, y_train)
search = tune_forest(X_train, y_train, n_iter=8)
tuned = search.best_estimator_
print(search.best_params_)


def auc(model, features, labels):
    return float(roc_auc_score(labels, model.predict_proba(features)[:, 1]))


tuned_train_auc = auc(tuned, X_train, y_train)
tuned_val_auc = auc(tuned, X_val, y_val)
logistic_val_auc = auc(logistic, X_val, y_val)
forest_val_auc = auc(forest, X_val, y_val)
print("logistic validation", round(logistic_val_auc, 3))
print("forest validation", round(forest_val_auc, 3))
print("tuned train", round(tuned_train_auc, 3))
print("tuned validation", round(tuned_val_auc, 3))


## Search `C`, then choose a model

`tune_forest` already searched trees, depth, and leaf size. Logistic regression still has `C`.

Search `0.1`, `1.0`, and `10.0` with `GridSearchCV` on the train rows only. Use `make_logistic_pipeline()`, `scoring="roc_auc"`, `cv=3`, and `n_jobs=1`. Fit it on `X_train` and `y_train`. Store the search in `search_c` and the best `C` in `best_C`.

Expect `best_C` of `0.1`. The best cross-validation ROC-AUC is near 0.78. That score is from train folds. The model you keep is still the one with the best validation AUC among logistic, forest, and the tuned forest.


In [ ]:
from sklearn.model_selection import GridSearchCV

from pipelines import make_logistic_pipeline

search_c = None
best_C = None


In [ ]:
if search_c is None or best_C is None:
    raise NotImplementedError("Search model__C with GridSearchCV. Use 0.1, 1, and 10.")
print(search_c.best_params_)
print("best C", best_C)
print("best cv AUC", round(float(search_c.best_score_), 3))


### Exercises

`beats_baseline` is True when the tuned validation AUC is above 0.5.

`gap` is the tuned train AUC minus the tuned validation AUC. A large gap means the train score is flattering. Write one sentence under the code about whether you would show this model to the team.

`chosen_model` is `"logistic"`, `"forest"`, or `"tuned_forest"`. Pick the highest validation AUC printed above. Do not use the test rows. The winner of those three is `logistic`.


In [ ]:
beats_baseline = None
gap = None
chosen_model = None


In [ ]:
if beats_baseline is None or gap is None or chosen_model is None:
    raise NotImplementedError("Finish the baseline, gap, and model choice.")
print("beats majority AUC", bool(beats_baseline))
print("train minus validation", round(float(gap), 3))
print("chosen", chosen_model)


Write one sentence here. Is the tuned validation score above 0.5, and is the gap small enough to keep?


## Checkpoint

Notebook 05 refits from the CSV, so you can close this kernel. Do not inspect test rows here.
